# Láminas y QR — *De Cádiz a Nueva York* (UCA, 18-XI-2026)

Genera, a partir de una tabla de láminas:
1. **Un QR por cuadro** (PNG, 300 ppp, con pie), que apunta a **nuestra** página en GitHub Pages, nunca directamente a un museo: si un enlace externo cambia, se corrige la página y el QR impreso sigue valiendo.
2. **La página `laminas/index.html`** del repositorio, con una ficha por cuadro: imagen (solo si los derechos lo permiten), procedencia resumida y enlace a la ficha oficial del museo.
3. **El Excel de control de derechos** (formato estándar).

Repositorio: https://github.com/JJR2025/De_Cadiz_a_Nueva_York · Web: https://jjr2025.github.io/De_Cadiz_a_Nueva_York/

Para añadir una imagen autorizada: copiarla en `laminas/img/` con el nombre de la columna `imagen` y volver a ejecutar.

*Nota de uso de IA: tabla, código y textos de las fichas preparados con Claude bajo la dirección de los autores, que fijan las ideas centrales y las conclusiones; IA con salvaguardas y transparencia.*

In [1]:
import pandas as pd, qrcode, html, os
from PIL import Image, ImageDraw, ImageFont
from datetime import date

BASE_URL = "https://jjr2025.github.io/De_Cadiz_a_Nueva_York/"
OUT = "laminas"
os.makedirs(f"{OUT}/qr", exist_ok=True)
os.makedirs(f"{OUT}/img", exist_ok=True)   # aquí van las imágenes que SÍ podemos alojar

In [2]:
# Estado de derechos (revisar antes de publicar).
# politica: "reproducir" = imagen en el texto y en la web; "enlazar" = solo QR a la ficha del museo;
#           "permiso" = pedir autorización; mientras tanto, solo enlace.
laminas = pd.DataFrame([
 dict(id="goya", n=1, obra="Goya, Retrato de Sebastián Martínez (1792)", sede="The Metropolitan Museum of Art, Nueva York (06.289)",
      ficha="https://www.metmuseum.org/art/collection/search/436541", imagen="goya_met_06289.jpg",
      derechos="Open Access del Met, CC0", politica="reproducir"),
 dict(id="cano", n=2, obra="Alonso Cano, Retrato de un eclesiástico, el «Inquisidor» (1625-1628)", sede="Hispanic Society of America, Nueva York (donación de A. M. Huntington, 1932)",
      ficha="https://hispanicsociety.emuseum.com/objects/963/portrait-of-an-ecclesiastic", imagen="", derechos="Por confirmar condiciones de la Hispanic Society", politica="permiso"),
 dict(id="murillo", n=3, obra="Murillo, boceto de los Desposorios místicos de Santa Catalina (c. 1680-1682)", sede="Los Angeles County Museum of Art (M.83.168), donación de The Ahmanson Foundation; 71,1 × 52,1 cm",
      ficha="https://collections.lacma.org/object/45745", imagen="murillo_lacma_M83168.jpg",
      derechos="Dominio público, descarga libre de LACMA", politica="reproducir"),
 dict(id="inza", n=4, obra="Joaquín Inza, Retrato de Sebastián Martínez (c. 1776)", sede="Colección particular. Sotheby's Nueva York, Arcade Fine Arts, 2004, lote 53; 98 × 72,8 cm",
      ficha="https://www.sothebys.com/en/auctions/ecatalogue/2004/arcade-fine-arts-old-master-to-contemporary-n01743/lot.53.html", imagen="inza_sothebys.jpg", derechos="Obra en dominio público; foto de catálogo de Sotheby's", politica="reproducir"),
 dict(id="rufina", n=5, obra="Velázquez, Santa Rufina (1629-1632)", sede="Fundación Focus, Sevilla",
      ficha="https://fundacionfocus.es/centro-velazquez-ant/santa-rufina/", imagen="",
      derechos="Pedir autorización a Focus (Cesión de imágenes; focus@fundacionfocus.com)", politica="permiso"),
 dict(id="abanico", n=6, obra="Goya, Mujer joven con abanico (c. 1806-1807)", sede="Musée du Louvre, París",
      ficha="https://fundaciongoyaenaragon.es/obra/mujer-joven-con-abanico/311", imagen="", derechos="Condiciones del Louvre / GrandPalaisRmn", politica="enlazar"),
 dict(id="resurreccion", n=7, obra="Atribuida a Solimena, Resurrección", sede="En venta en 1stDibs",
      ficha="", imagen="", derechos="Foto del anunciante", politica="enlazar"),
 dict(id="giordano", n=8, obra="Luca Giordano, La vocación de San Mateo (c. 1685)", sede="Georgetown University Art Collection, Washington (comprado a Martha Meade, 1860)",
      ficha="https://library.georgetown.edu/special-collections/art/guac/Giordano", imagen="",
      derechos="Condiciones de Georgetown University Library", politica="enlazar"),
 dict(id="cano_nino", n=9, obra="Atribuido a Alonso Cano, El Niño Jesús dormido (c. 1652)", sede="Kingston Lacy, The Bankes Collection, National Trust (NT 1257133)",
      ficha="https://www.nationaltrustcollections.org.uk/object/1257133", imagen="",
      derechos="© National Trust Images", politica="enlazar"),
])
laminas["url_qr"] = BASE_URL + "laminas/#" + laminas["id"]
laminas

,id,n,obra,sede,ficha,imagen,derechos,politica,url_qr
0,goya,1,"Goya, Retrato de Sebastián Martínez (1792)","The Metropolitan Museum of Art, Nueva York (06...",https://www.metmuseum.org/art/collection/searc...,goya_met_06289.jpg,"Open Access del Met, CC0",reproducir,https://jjr2025.github.io/De_Cadiz_a_Nueva_Yor...
1,cano,2,"Alonso Cano, Retrato de un eclesiástico, el «I...","Hispanic Society of America, Nueva York (donac...",https://hispanicsociety.emuseum.com/objects/96...,,Por confirmar condiciones de la Hispanic Society,permiso,https://jjr2025.github.io/De_Cadiz_a_Nueva_Yor...
2,murillo,3,"Murillo, boceto de los Desposorios místicos de...","Los Angeles County Museum of Art (M.83.168), d...",https://collections.lacma.org/object/45745,murillo_lacma_M83168.jpg,"Dominio público, descarga libre de LACMA",reproducir,https://jjr2025.github.io/De_Cadiz_a_Nueva_Yor...
3,inza,4,"Joaquín Inza, Retrato de Sebastián Martínez (c...","Colección particular. Sotheby's Nueva York, Ar...",https://www.sothebys.com/en/auctions/ecatalogu...,inza_sothebys.jpg,Obra en dominio público; foto de catálogo de S...,reproducir,https://jjr2025.github.io/De_Cadiz_a_Nueva_Yor...
4,rufina,5,"Velázquez, Santa Rufina (1629-1632)","Fundación Focus, Sevilla",https://fundacionfocus.es/centro-velazquez-ant...,,Pedir autorización a Focus (Cesión de imágenes...,permiso,https://jjr2025.github.io/De_Cadiz_a_Nueva_Yor...
5,abanico,6,"Goya, Mujer joven con abanico (c. 1806-1807)","Musée du Louvre, París",https://fundaciongoyaenaragon.es/obra/mujer-jo...,,Condiciones del Louvre / GrandPalaisRmn,enlazar,https://jjr2025.github.io/De_Cadiz_a_Nueva_Yor...
6,resurreccion,7,"Atribuida a Solimena, Resurrección",En venta en 1stDibs,,,Foto del anunciante,enlazar,https://jjr2025.github.io/De_Cadiz_a_Nueva_Yor...
7,giordano,8,"Luca Giordano, La vocación de San Mateo (c. 1685)","Georgetown University Art Collection, Washingt...",https://library.georgetown.edu/special-collect...,,Condiciones de Georgetown University Library,enlazar,https://jjr2025.github.io/De_Cadiz_a_Nueva_Yor...
8,cano_nino,9,"Atribuido a Alonso Cano, El Niño Jesús dormido...","Kingston Lacy, The Bankes Collection, National...",https://www.nationaltrustcollections.org.uk/ob...,,© National Trust Images,enlazar,https://jjr2025.github.io/De_Cadiz_a_Nueva_Yor...


In [3]:
def hacer_qr(url, titulo, sub, path):
    q = qrcode.QRCode(error_correction=qrcode.constants.ERROR_CORRECT_H, box_size=20, border=4)
    q.add_data(url); q.make(fit=True)
    img = q.make_image(fill_color="black", back_color="white").convert("RGB")
    w, h = img.size
    c = Image.new("RGB", (w, h + 150), "white"); c.paste(img, (0, 0))
    d = ImageDraw.Draw(c)
    try:
        f1 = ImageFont.truetype("DejaVuSans-Bold.ttf", 38); f2 = ImageFont.truetype("DejaVuSans.ttf", 30)
    except OSError:
        f1 = f2 = ImageFont.load_default()
    for t, f, y in [(titulo, f1, h), (sub, f2, h + 60)]:
        t = t if d.textlength(t, font=f) < w - 20 else t[:int(len(t) * (w - 40) / d.textlength(t, font=f))] + "…"
        d.text(((w - d.textlength(t, font=f)) / 2, y), t, fill="black", font=f)
    c.save(path, dpi=(300, 300))

for r in laminas.itertuples():
    hacer_qr(r.url_qr, f"Lámina {r.n}", r.obra.split(",")[0], f"{OUT}/qr/QR_{r.n:02d}_{r.id}.png")
sorted(os.listdir(f"{OUT}/qr"))

['QR_01_goya.png',
 'QR_02_cano.png',
 'QR_03_murillo.png',
 'QR_04_inza.png',
 'QR_05_rufina.png',
 'QR_06_abanico.png',
 'QR_07_resurreccion.png',
 'QR_08_giordano.png',
 'QR_09_cano_nino.png']

In [4]:
# Página de láminas para GitHub Pages
fichas = []
for r in laminas.itertuples():
    tiene_img = r.politica == "reproducir" and r.imagen and os.path.exists(f"{OUT}/img/{r.imagen}")
    fig = (f'<img src="img/{html.escape(r.imagen)}" alt="{html.escape(r.obra)}">' if tiene_img
           else '<div class="vacio">Imagen disponible en la ficha de la institución</div>')
    enlace = (f'<p><a href="{html.escape(r.ficha)}">Ver y descargar en la ficha oficial</a><br>'
              f'<span class="url">{html.escape(r.ficha)}</span></p>') if r.ficha else ""
    fichas.append(f"""<section id="{r.id}"><p class="n">Lámina {r.n}</p>{fig}
<h2>{html.escape(r.obra)}</h2><p>{html.escape(r.sede)}</p><p class="d">{html.escape(r.derechos)}</p>{enlace}</section>""")

pagina = f"""<!doctype html><html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>Láminas · De Cádiz a Nueva York</title>
<style>body{{font-family:Georgia,serif;max-width:760px;margin:auto;padding:16px;color:#1E2329;background:#F6F3EC}}
h1{{color:#0B769F}}section{{border-top:2px solid #0B769F;padding:24px 0}}img{{max-width:100%}}
.vacio{{border:1px dashed #8A939C;padding:48px;text-align:center;color:#4A5560}}.n{{text-transform:uppercase;letter-spacing:2px;color:#0B769F}}
.d{{font-size:.85em;color:#4A5560}}.url{{font-family:Arial,sans-serif;font-size:.8em;color:#4A5560;word-break:break-all}}a{{color:#0B769F}}</style></head><body>
<h1>De Cádiz a Nueva York: seis cuadros de Sebastián Martínez</h1>
<p>Lesley Crewdson y José Juan Ruiz · Universidad de Cádiz, 18 de noviembre de 2026</p>
{''.join(fichas)}
<p class="d">Las imágenes se reproducen solo cuando la institución titular las ofrece en dominio público o con autorización; en los demás casos se remite a su ficha oficial. Actualizado: {date.today():%d-%m-%Y}.</p>
</body></html>"""
open(f"{OUT}/index.html", "w", encoding="utf-8").write(pagina)
print("OK", f"{OUT}/index.html")

OK laminas/index.html


In [5]:
# Excel de control (formato estándar: CircularXX Book 11, ancho 15 centrado, cabecera #0B769F)
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
os.makedirs("control", exist_ok=True); xlsx = f"control/control_derechos_laminas_{date.today():%Y%m%d}.xlsx"
laminas.to_excel(xlsx, index=False, sheet_name="Láminas")
wb = load_workbook(xlsx); ws = wb.active
for col in ws.columns:
    ws.column_dimensions[col[0].column_letter].width = 15
    for c in col:
        c.font = Font(name="CircularXX Book", size=11)
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
for c in ws[1]:
    c.font = Font(name="CircularXX Book", size=11, bold=True, color="FFFFFF")
    c.fill = PatternFill("solid", fgColor="0B769F")
wb.save(xlsx); print(xlsx)

control/control_derechos_laminas_20261024.xlsx
